# Module 3 Homework: Data Warehousing & BigQuery

 ### Loading the data
---
Python script: load_yellow_taxi_data.py

### Question 1. Counting records
---

### What is count of records for the 2024 Yellow Taxi Data?


1. GCS Upload: Successfully uploaded 6 Parquet files (Jan-Jun 2024) to the bucket de-zoomcamp--2026.

2. Dataset Creation: Created a BigQuery dataset named trips_data_all in the us-west1 region.

3. External Table: Created an External Table named external_yellow_tripdata pointing to the GCS Parquet files.

4. Native Table: Created a Native Table named yellow_tripdata_2024_non_partitioned by importing the data into BigQuery's internal storage.

#### Query:

```sql
SELECT count(*) 
FROM `de-zoomcamp-487123.trips_data_all.external_yellow_tripdata`;
```
* Result: **20,332,093**

### Question 2. Data read estimation
---

Write a query to count the distinct number of PULocationIDs for the entire dataset on both the tables.

What is the estimated amount of data that will be read when this query is executed on the External Table and the Table?



Steps：
1. Create Native Table: Use SQL to move data from GCS to BigQuery storage.

#### Query:
```sql
CREATE OR REPLACE TABLE `de-zoomcamp-487123.trips_data_all.yellow_tripdata_materialized` 
AS 
SELECT * FROM `de-zoomcamp-487123.trips_data_all.external_yellow_tripdata`;
```
2. Test External Table: Paste the count query and check the top right estimate.

* Result: 0 B (BigQuery can't estimate external files).

3. Test Native Table: Paste the same query for the materialized table.

* Result: 155.12 MB (BigQuery knows the column size internally).

**Answer**: 0 MB for the External Table and 155.12 MB for the Materialized Table.

### Question 3. Understanding columnar storage
---

Write a query to retrieve the PULocationID from the table (not the external table) in BigQuery. Now write a query to retrieve the PULocationID and DOLocationID on the same table.

Why are the estimated number of Bytes different?

Query One Column:
```sql
SELECT PULocationID FROM `de-zoomcamp-487123.trips_data_all.yellow_tripdata_materialized`;
```
* **Result**: 155.12 MB.

Query Two Columns:
```sql
SELECT PULocationID, DOLocationID FROM `de-zoomcamp-487123.trips_data_all.yellow_tripdata_materialized`;
```
* **Result**: 310.24 MB (Exactly double!).

* **Conclusion**: BigQuery only scans requested columns; more columns = more data processed.

**Answer**： BigQuery is a columnar database, and it only scans the specific columns requested in the query. Querying two columns (PULocationID, DOLocationID) requires reading more data than querying one column (PULocationID), leading to a higher estimated number of bytes processed.

### Question 4. Counting zero fare trips
---

How many records have a fare_amount of 0?

```sql
SELECT count(*)
FROM `de-zoomcamp-487123.trips_data_all.yellow_tripdata_materialized`
WHERE fare_amount = 0;
```
* **Result**: 8,333

### Question 5. Partitioning and clustering
---

What is the best strategy to make an optimized table in Big Query if your query will always filter based on tpep_dropoff_datetime and order the results by VendorID (Create a new table with this strategy)


```sql
CREATE OR REPLACE TABLE `de-zoomcamp-487123.trips_data_all.yellow_tripdata_partitioned_clustered`
PARTITION BY DATE(tpep_dropoff_datetime)
CLUSTER BY VendorID AS
SELECT * FROM `de-zoomcamp-487123.trips_data_all.external_yellow_tripdata`;
```

* **Result**: Partition by tpep_dropoff_datetime and Cluster on VendorID

### Question 6. Partition benefits
---

Write a query to retrieve the distinct VendorIDs between tpep_dropoff_datetime 2024-03-01 and 2024-03-15 (inclusive)

Use the materialized table you created earlier in your from clause and note the estimated bytes. Now change the table in the from clause to the partitioned table you created for question 5 and note the estimated bytes processed. What are these values?

Choose the answer which most closely matches.

Step 1: Query the Non-partitioned Table
Action: Run a query with a WHERE filter for a 15-day range on the Native Table we created first.

```sql
SELECT DISTINCT(VendorID)
FROM `de-zoomcamp-487123.trips_data_all.yellow_tripdata_materialized`
WHERE tpep_dropoff_datetime BETWEEN '2024-03-01' AND '2024-03-15';

Step 2: Query the Partitioned Table
Action: Run the exact same query on the new table we created with PARTITION BY.

```sql
SELECT DISTINCT(VendorID)
FROM `de-zoomcamp-487123.trips_data_all.yellow_tripdata_partitioned_clustered`
WHERE tpep_dropoff_datetime BETWEEN '2024-03-01' AND '2024-03-15';
```

* **Result**: The "Partitioned" table reduces the scan size from 310 MB to 26 MB.

### Question 7. External table storage
---

Where is the data stored in the External Table you created?

Internal Table (Native): Data is stored inside BigQuery's managed storage.

External Table: Data remains outside in Google Cloud Storage (GCS) buckets.

**Answer**: Since it's an "External" table, the data is stored in the **GCP Bucket**.



### Question 8.
---

It is best practice in Big Query to always cluster your data: (True/False)

Clustering is best for large data (over 1 GB). For small tables, the effort to organize the data is more work than the reward. So, don't use it for everything

**Answer**: False


### Question 9. Understanding table scans
---

No Points: Write a SELECT count(*) query FROM the materialized table you created. How many bytes does it estimate will be read? Why?

**Estimated Bytes: 0 B**

Because BigQuery reads the total row count from the Table Metadata, not from the actual data storage


